# Tolkien Anchored Sentiment — Held-out Model Evaluation

This notebook evaluates an anchor-based sentiment scoring procedure on a held-out set of manually labelled sentences from *The Lord of the Rings*.

The procedure compares each test sentence with positive and negative reference examples, called **anchor sentences** in this notebook. The prediction is assigned according to the closer anchor centroid.

This is a nearest-centroid evaluation procedure. It should not be confused with the semantic-axis projection method used in the separate semantic-axis notebook.

**Core idea**

1. Build a positive anchor centroid and a negative anchor centroid from anchor sentences.
2. Embed each held-out test sentence.
3. Compare each test sentence to both centroids using cosine similarity.
4. Predict `positive` or `negative` according to the closer centroid.
5. Compute evaluation metrics: confusion matrix, accuracy, precision, recall, and F1.

Anchor and test sentences are kept disjoint to avoid leakage between the scoring setup and the evaluation set.

In [ ]:
# Imports

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sentence_transformers import SentenceTransformer

In [ ]:
# Add local src path

PROJECT_ROOT = Path("..").resolve()
SRC_PATH = PROJECT_ROOT / "src"

if str(SRC_PATH) not in sys.path:
    sys.path.append(str(SRC_PATH))

In [ ]:
# Import evaluation utilities

from literary_nlp.evaluation import (
    evaluate_anchor_classifier,
    mcnemar_contingency_table,
    validate_anchor_eval_dataframe,
)

## Input data format

The manually selected sentences are divided into two roles:

1. **Anchor sentences**  
   These are short positive and negative reference sentences used by the sentiment-scoring procedure. They are not treated as test examples, because they contribute directly to the scoring setup.

2. **Gold-labelled test sentences**  
   These are held-out positive and negative examples used only for evaluation. They are not used as anchors. This separation avoids evaluating the models on sentences that were already used as reference examples.

In other words, the anchor sentences provide the positive and negative reference points for the method, while the test sentences provide an independent check of whether the resulting model-based sentiment scores align with manually assigned labels.

In [ ]:
# Cell — Load evaluation data

DATA_PATH = Path("data/eval_sentences.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {DATA_PATH}. "
        "For the public repository, this file is expected to be supplied locally "
        "because it may contain copyrighted text."
    )

df = pd.read_csv(DATA_PATH, sep=";", engine="python")
df = validate_anchor_eval_dataframe(df)

anchors = df[df["split"] == "anchor"].copy()
test = df[df["split"] == "test"].copy()

print("\nRows used by role:")
print(f"Anchors: {len(anchors)}")
print(f"Test:    {len(test)}")

print("\nGold-label counts:")
print(df["gold_label"].value_counts())

display(df.head(10))

## Choose models

This notebook compares a general-purpose sentence-transformer baseline with a Tolkien-adapted sentence-transformer model.

The adapted model path is expected to point to a local model directory. The model itself is not included in this repository. Users can either update the path to their own local model or replace it with another SentenceTransformer-compatible model.

In [ ]:
# Cell — Choose models

MODEL_CONFIGS = {
    "MiniLM baseline": {
        "path": "sentence-transformers/all-MiniLM-L6-v2",
        "type": "huggingface",
    },
    "Tolkien-adapted model": {
        "path": Path("../models/tolkien_sentence_transformer_epoch_1"),
        "type": "local",
    },
}


def load_model(model_name: str, config: dict) -> SentenceTransformer:
    """
    Load a SentenceTransformer model from Hugging Face or from a local path.
    """
    model_path = config["path"]

    if config["type"] == "local" and not Path(model_path).exists():
        raise FileNotFoundError(
            f"Local model path not found for '{model_name}': {model_path}\n"
            "Update MODEL_CONFIGS with the correct local path before running this notebook."
        )

    print(f"Loading model: {model_name}")
    print(f"Path: {model_path}")

    return SentenceTransformer(str(model_path))


base_model = load_model(
    "MiniLM baseline",
    MODEL_CONFIGS["MiniLM baseline"],
)

tolkien_model = load_model(
    "Tolkien-adapted model",
    MODEL_CONFIGS["Tolkien-adapted model"],
)

## Run evaluation for individual models

The following cells apply the same evaluation function to the MiniLM baseline and the Tolkien-adapted model. Using a shared function ensures that both models are evaluated with the same anchors, test set, and scoring procedure.

In [ ]:
# Cell — Evaluate MiniLM baseline

base_result = evaluate_anchor_classifier(
    model=base_model,
    anchors=anchors,
    test=test,
    batch_size=64,
)

base_result.metrics

In [ ]:
# Cell — Confusion matrix for MiniLM baseline

base_result.confusion

In [ ]:
# Cell — Classification report for MiniLM baseline

base_result.classification_report_df

In [ ]:
# Cell — Misclassified sentences: MiniLM baseline

base_errors = (
    base_result.predictions[
        base_result.predictions["pred_label"]
        != base_result.predictions["gold_label"]
    ]
    .sort_values("margin_positive_minus_negative")
)

base_errors[
    [
        "sentence_text",
        "gold_label",
        "pred_label",
        "similarity_positive",
        "similarity_negative",
        "margin_positive_minus_negative",
    ]
]

## Evaluate Tolkien-adapted model

This section evaluates the locally configured Tolkien-adapted model using the same anchor-based evaluation procedure as the MiniLM baseline.

In [ ]:
# Cell — Evaluate Tolkien-adapted model

tolkien_result = evaluate_anchor_classifier(
    model=tolkien_model,
    anchors=anchors,
    test=test,
    batch_size=64,
)

tolkien_result.metrics

In [ ]:
# Confusion matrix for the Tolkien-adapted model

tolkien_result.confusion

In [ ]:
# Cell — Classification report for the Tolkien-adapted model

tolkien_result.classification_report_df

In [ ]:
# Cell — Misclassified sentences: Tolkien-adapted model

tolkien_errors = (
    tolkien_result.predictions[
        tolkien_result.predictions["pred_label"]
        != tolkien_result.predictions["gold_label"]
    ]
    .sort_values("margin_positive_minus_negative")
)

tolkien_errors[
    [
        "sentence_text",
        "gold_label",
        "pred_label",
        "similarity_positive",
        "similarity_negative",
        "margin_positive_minus_negative",
    ]
]

## Baseline vs Tolkien-adapted model

In [ ]:
# Cell — Compare MiniLM baseline and Tolkien-adapted model

comparison_df = pd.DataFrame(
    [
        {
            "model": "MiniLM baseline",
            **base_result.metrics,
        },
        {
            "model": "Tolkien-adapted model",
            **tolkien_result.metrics,
        },
    ]
)

comparison_df

In [ ]:
# Cell — Plot selected metric comparison

metrics_to_plot = [
    "accuracy",
    "precision_positive",
    "recall_positive",
    "f1_positive",
]

comparison_df.set_index("model")[metrics_to_plot].T.plot(
    kind="bar",
    figsize=(8, 4),
)

plt.ylabel("Score")
plt.ylim(0, 1)
plt.title("Model comparison on held-out evaluation sentences")
plt.xticks(rotation=45, ha="right")
plt.legend(title="Model")
plt.tight_layout()
plt.show()

## Epoch comparison for Tolkien-adapted models

This section evaluates Tolkien-adapted models fine-tuned for different numbers of epochs. The comparison is used to identify the strongest adapted model before performing the paired comparison with the MiniLM baseline.

In [ ]:
# Cell — Evaluate Tolkien-adapted models across epochs

EPOCH_MODEL_CONFIGS = {
    "Tolkien 1 epoch": Path("../models/tolkien_sentence_transformer_epoch_1"),
    "Tolkien 2 epochs": Path("../models/tolkien_sentence_transformer_epoch_2"),
    "Tolkien 4 epochs": Path("../models/tolkien_sentence_transformer_epoch_4"),
    "Tolkien 8 epochs": Path("../models/tolkien_sentence_transformer_epoch_8"),
}

epoch_results = {}

for model_name, model_path in EPOCH_MODEL_CONFIGS.items():
    if not model_path.exists():
        raise FileNotFoundError(
            f"Model path not found for {model_name}: {model_path}"
        )

    print(f"Loading and evaluating: {model_name}")
    model = SentenceTransformer(str(model_path))

    epoch_results[model_name] = evaluate_anchor_classifier(
        model=model,
        anchors=anchors,
        test=test,
        batch_size=64,
    )

In [ ]:
# Cell — Summarise Tolkien epoch results

epoch_comparison_df = pd.DataFrame(
    [
        {
            "model": model_name,
            **result.metrics,
        }
        for model_name, result in epoch_results.items()
    ]
)

epoch_comparison_df

In [ ]:
# Cell — Compact epoch comparison table

epoch_metric_cols = [
    "model",
    "accuracy",
    "precision_positive",
    "recall_positive",
    "f1_positive",
    "precision_negative",
    "recall_negative",
    "f1_negative",
]

epoch_comparison_df[epoch_metric_cols]

In [ ]:
# Cell — Plot Tolkien epoch comparison

metrics_to_plot = [
    "accuracy",
    "precision_positive",
    "recall_positive",
    "f1_positive",
]

epoch_comparison_df.set_index("model")[metrics_to_plot].plot(
    kind="bar",
    figsize=(10, 5),
)

plt.ylabel("Score")
plt.ylim(0, 1)
plt.title("Tolkien-adapted model comparison across fine-tuning epochs")
plt.xticks(rotation=45, ha="right")
plt.legend(title="Metric")
plt.tight_layout()
plt.show()

In [ ]:
# Cell — Select Tolkien-adapted model for final comparison

SELECTED_MODEL_NAME = "Tolkien 1 epoch"

selected_tolkien_result = epoch_results[SELECTED_MODEL_NAME]
selected_tolkien_path = EPOCH_MODEL_CONFIGS[SELECTED_MODEL_NAME]

selected_tolkien_model = SentenceTransformer(str(selected_tolkien_path))

print(f"Selected model for final comparison: {SELECTED_MODEL_NAME}")

In [ ]:
# Cell — Compare MiniLM baseline and selected Tolkien-adapted model

baseline_vs_selected_df = pd.DataFrame(
    [
        {
            "model": "MiniLM baseline",
            **base_result.metrics,
        },
        {
            "model": SELECTED_MODEL_NAME,
            **selected_tolkien_result.metrics,
        },
    ]
)

baseline_vs_selected_df

## Paired model comparison

McNemar's test is used here because both models are evaluated on the same held-out test sentences. The test compares whether the two models make different errors on the same examples, rather than only comparing aggregate accuracy scores.

After comparing the epoch-specific Tolkien models, the selected adapted model is compared with the MiniLM baseline.

In [ ]:
# Cell — McNemar's test: MiniLM baseline vs selected Tolkien-adapted model

from statsmodels.stats.contingency_tables import mcnemar

mcnemar_table_df = mcnemar_contingency_table(
    y_true=test["gold_label"].to_numpy(),
    first_predictions=base_result.predictions["pred_label"].to_numpy(),
    second_predictions=selected_tolkien_result.predictions["pred_label"].to_numpy(),
    first_name="MiniLM baseline",
    second_name=SELECTED_MODEL_NAME,
)

display(mcnemar_table_df)

mcnemar_table = mcnemar_table_df.to_numpy().tolist()
mcnemar_result = mcnemar(mcnemar_table, exact=True)

print(f"\nMcNemar p-value: {mcnemar_result.pvalue:.4f}")

if mcnemar_result.pvalue < 0.05:
    print("Result: The difference between the two models is statistically significant.")
else:
    print("Result: The difference between the two models is not statistically significant.")

## Qualitative retrieval comparison

This section compares nearest-neighbour retrieval results between the MiniLM baseline model and the selected Tolkien-adapted model.

The retrieval test is qualitative. It is used to inspect whether the adapted model retrieves sentences that are semantically closer to the query within the literary domain. It is not used as a classification metric.

The full corpus text is not included in this repository because it contains copyrighted material. To run this section locally, provide a sentence-per-line corpus file and update `CORPUS_PATH`.

In [ ]:
# Cell — Retrieval helper

def retrieve_similar_sentences(
    model: SentenceTransformer,
    corpus_sentences: list[str],
    query: str,
    top_k: int = 10,
    batch_size: int = 64,
) -> pd.DataFrame:
    """
    Retrieve the most similar corpus sentences to a query sentence.

    Embeddings are normalised, so cosine similarity is computed as a dot product.
    """
    if not corpus_sentences:
        raise ValueError("corpus_sentences is empty.")

    corpus_embeddings = model.encode(
        corpus_sentences,
        batch_size=batch_size,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True,
    )

    query_embedding = model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True,
    )[0]

    similarities = corpus_embeddings @ query_embedding

    top_indices = np.argsort(similarities)[::-1][:top_k]

    return pd.DataFrame(
        {
            "rank": range(1, len(top_indices) + 1),
            "sentence": [corpus_sentences[i] for i in top_indices],
            "similarity": [similarities[i] for i in top_indices],
        }
    )

In [ ]:
# Cell — Load retrieval corpus

RETRIEVAL_CORPUS_PATH = Path("data/LotR.txt")

if not RETRIEVAL_CORPUS_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {RETRIEVAL_CORPUS_PATH}. "
        "The retrieval corpus is not included in the public repository because it contains copyrighted text."
    )

with open(RETRIEVAL_CORPUS_PATH, "r", encoding="utf-8") as file:
    corpus_sentences = [
        line.strip()
        for line in file
        if line.strip()
    ]

print(f"Loaded {len(corpus_sentences)} retrieval sentences.")

In [ ]:
# Cell — Compare retrieval results

QUERY_SENTENCE = "As they came out again into the open country a great wind came."
TOP_K = 10

base_retrieval_results = retrieve_similar_sentences(
    model=base_model,
    corpus_sentences=corpus_sentences,
    query=QUERY_SENTENCE,
    top_k=TOP_K,
)

tolkien_retrieval_results = retrieve_similar_sentences(
    model=selected_tolkien_model,
    corpus_sentences=corpus_sentences,
    query=QUERY_SENTENCE,
    top_k=TOP_K,
)

base_retrieval_results

In [ ]:
# Cell — Tolkien-adapted retrieval results

tolkien_retrieval_results

In [ ]:
# Cell — Compare retrieval rankings

retrieval_comparison = pd.DataFrame(
    {
        "rank": base_retrieval_results["rank"],
        "MiniLM sentence": base_retrieval_results["sentence"],
        "MiniLM similarity": base_retrieval_results["similarity"],
        "Tolkien-adapted sentence": tolkien_retrieval_results["sentence"],
        "Tolkien-adapted similarity": tolkien_retrieval_results["similarity"],
    }
)

retrieval_comparison